# LLM inference and Unsloth LoRA fine-tuning (exploratory)

**Source:** Part 3 of the original master notebook (cells 99-117). *This workflow is not one of the three requested task notebooks;* it is kept as a separate
notebook so that no implemented work is silently dropped.

## 1. Objective and scope
1. Zero-shot prompt of `Qwen/Qwen2.5-0.5B-Instruct` to produce 3 emotion keywords and Big Five estimates for one PANDORA text (one example, no evaluation).
2. LoRA fine-tuning (Unsloth, 4-bit `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`) so that, given a French sentence, the model outputs the name of a Big Five trait.
3. A qualitative check on five hand-written French sentences.

**What the source does *not* contain:** any quantitative evaluation, held-out set, or comparison with the other models. The five test sentences were written by the author with the
expected trait in mind and all five were answered as expected in the recorded run - this is an anecdotal sanity check, not a measured accuracy. The training file
`personality_emotions_french_no_labels.jsonl` (3,400 rows; columns include `text` and `personality`) is private and not distributed.

**Original errors left out:** cells 110-111 (first SFT attempt using `input/output` columns) failed in the source (`NameError: DATA_DIR`, then `RuntimeError`);
only the working path (cells 109 → 114 → 116 → 117) is reproduced here.

**Environment:** needs a CUDA GPU on Linux. Original: Colab A100-80GB, Unsloth 2026.10.1, transformers 4.57.6, torch 2.11.0+cu130, bitsandbytes 0.50.2, wandb logged in.
The TRL version is not recorded; the `SFTTrainer(dataset_text_field=..., max_seq_length=...)` signature below is the original's and may need adapting to your TRL version.
**Not executed in the preparation environment.** Install: `pip install -r requirements-llm.txt`.

In [ ]:
import sys
from pathlib import Path

# Locate the repository root (works from the repo root or from notebooks/) and make `src` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").is_file())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import get_paths, get_device, set_seed, save_table, save_figure

paths = get_paths(create=True)      # data/, results/, cache/, checkpoints/ (override with ITC_* env vars)
SEED = 42
set_seed(SEED)                      # seed once, as the original notebook did
DEVICE = get_device()
print("repo root:", ROOT, "| device:", DEVICE)

## 2. Zero-shot inference (Qwen2.5-0.5B-Instruct)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.models.llm import build_ocean_analysis_messages

SAMPLE_TEXT = "Paste any short text here."   # the original used PANDORA test row 42 (cleaned text); not copied into the repo
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

chat_text = tokenizer.apply_chat_template(build_ocean_analysis_messages(SAMPLE_TEXT), tokenize=False, add_generation_prompt=True)
inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)
generated = model.generate(inputs.input_ids, max_new_tokens=150, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.batch_decode([g[len(i):] for i, g in zip(inputs.input_ids, generated)], skip_special_tokens=True)[0].strip())

## 3. LoRA fine-tuning with Unsloth (French personality-trait classification as text generation)
Config from the source: `max_seq_length=2048`, 4-bit, LoRA `r=16, alpha=16, dropout=0`, all attention+MLP projections, gradient checkpointing `"unsloth"`, `random_state=3407`;
SFT: batch 2 × grad-accum 4, 1 epoch (425 steps for 3,400 rows), lr 2e-4, `adamw_8bit`, weight-decay 0.01, linear schedule, warm-up 5 steps, seed 42.

*Deviation:* `report_to="none"` is added - the original left the default, which logged the run to a personal Weights & Biases account (visible in its output). It affects logging only.

In [ ]:
if DEVICE != "cuda":
    raise RuntimeError("Unsloth / bitsandbytes require a CUDA GPU on Linux.")

from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from src.models.llm import format_chatml_examples

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
                                                     max_seq_length=max_seq_length, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
    random_state=3407, use_rslora=False, loftq_config=None)

jsonl = paths.data_dir / "personality_emotions_french_no_labels.jsonl"
if not jsonl.is_file():
    raise FileNotFoundError(f"{jsonl} not found - this private file is not distributed (see README).")
raw = load_dataset("json", data_files=str(jsonl), split="train")
dataset_traits = raw.map(lambda ex: {"formatted_text": format_chatml_examples(ex["text"], ex["personality"])},
                         batched=True, remove_columns=raw.column_names)
print(len(dataset_traits), dataset_traits[0]["formatted_text"])

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset_traits, dataset_text_field="formatted_text",
    formatting_func=lambda x: x["formatted_text"] if isinstance(x["formatted_text"], list) else [x["formatted_text"]],
    max_seq_length=max_seq_length, dataset_num_proc=2, packing=False,
    args=TrainingArguments(per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=5, num_train_epochs=1,
                           learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=10,
                           optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=42,
                           output_dir=str(paths.checkpoints_dir / "unsloth_traits_outputs"), report_to="none"))
trainer_stats = trainer.train()

## 4. Qualitative check (five hand-written sentences - anecdotal, not an evaluation)

In [ ]:
from src.models.llm import chatml_inference_prompt

FastLanguageModel.for_inference(model)
tests = [("Je planifie tout à l'avance et je respecte les délais. Je suis très organisé.", "Conscientiousness"),
         ("Je m'inquiète pour un rien et je stresse très facilement face à l'imprévu.", "Neuroticism"),
         ("J'adore les soirées animées, rencontrer plein de monde et parler fort !", "Extraversion"),
         ("Je suis fasciné par l'art abstrait, la poésie et la découverte de nouvelles cultures.", "Openness"),
         ("J'essaie toujours d'aider les autres et de faire preuve de compassion et d'empathie.", "Agreeableness")]
for text, expected in tests:
    enc = tokenizer([chatml_inference_prompt(text)], return_tensors="pt").to("cuda")
    out = model.generate(**enc, max_new_tokens=15, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.batch_decode(out, skip_special_tokens=True)[0].split("assistant\n")[-1].strip()
    print(f"{answer:20s} (expected {expected})  <- {text}")

## 8. Interpretation
Recorded in the source: the five sentences above were each answered with the expected trait name. With five author-written, prototypical sentences, no held-out data and no baseline,
this says only that the fine-tuned model learned the output format and the obvious lexical cues of the training set; it provides no estimate of accuracy on real text.
The training data is synthetic-looking ("no_labels" file with a `personality` column) and its provenance/licence is not documented in the source.